# Chapitre 4 · Construire un moteur d'autograd (exercice)

Notebook du chapitre 4 de *Construire un LLM de zéro*. La machine qui se cache
derrière le `loss.backward()` du chapitre 1 : une classe `Value` qui enregistre
chaque calcul dans un graphe, un moteur d'autograd complet, validé par gradient
check, un neurone entraîné avec, et la preuve chiffrée que PyTorch fait pareil.

**Comment travailler.** D'abord la leçon : tout le code du chapitre, complet et
exécutable de bout en bout. Lis, exécute, modifie pour voir. Ensuite, la
section **Exercices** : c'est LÀ que tu écris TON moteur, trou par trou, validé
par des `assert`. **Le pacte IA débranchée s'applique à ces exercices en
entier** : ferme l'onglet de ton assistant IA, littéralement. Aucune IA n'écrit
ce code à ta place ; elle relit, vérifie et débloque une fois les asserts
passés, elle n'écrit pas.

Tout tourne **hors ligne, sans GPU** : Python pur (+ PyTorch, fourni, pour la
comparaison finale).

## 1. Le dernier verrou : `loss.backward()`

Une ligne porte toute la magie depuis le chapitre 1. Derrière elle : un moteur
d'**autograd**, un programme qui calcule tout seul les gradients de n'importe
quel calcul. Le tien tiendra en une soixantaine de lignes ; cette leçon les
construit sous tes yeux, et les exercices te les feront écrire de tes mains.

## 2. Pourquoi le capteur ne suffit plus

Le capteur du chapitre 3 mesure les pentes une par une, en réévaluant tout le
calcul à chaque fois : 2 × 91 497 évaluations complètes de MiniLM pour UN pas
de descente. L'autograd les obtient toutes en un seul aller-retour, grâce à la
mémoire du calcul : le **graphe**. On garde pourtant le capteur sous la main :
il sera notre juge indépendant pour tout le chapitre.

In [ ]:
import math


def pente(f, x, h=1e-5):
    """Le capteur du chapitre 3 : de combien f(x) bouge quand on pousse x d'un cheveu."""
    return (f(x + h) - f(x - h)) / (2 * h)


# Verification rapide : sur f(x) = x**2, la pente exacte en x = 3 vaut 6.
print(f"pente(x**2, en 3) = {pente(lambda x: x ** 2, 3.0):.6f}")

## 3. `Value` : une boîte qui se souvient

Un `float` est nu : `-6.0` ne sait pas qu'il est né d'un `2.0 * -3.0`. La boîte
`Value` transporte la valeur (`data`) ET sa généalogie : ses parents (`_prev`)
et l'opération qui l'a produite (`_op`). Les méthodes spéciales `__add__` et
`__mul__` (ce que Python appelle quand il voit `+` et `*`) calculent ET
déclarent la naissance.

In [ ]:
class Value:
    def __init__(self, data, _prev=(), _op=""):
        self.data = data          # la valeur, un float ordinaire
        self._prev = set(_prev)   # ses parents dans le graphe
        self._op = _op            # l'operation qui l'a produite

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), "+")

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), "*")

    def __repr__(self):
        return f"Value(data={self.data})"

Premier graphe : le calcul donne 4.0, comme n'importe quelle calculette, mais
rien n'est perdu. En suivant les `_prev` de proche en proche, on reconstitue
tout l'arbre généalogique du résultat.

In [ ]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
d = a * b        # -6.0, ne de a et b par "*"
e = d + c        #  4.0, ne de d et c par "+"

print(e)                                        # Value(data=4.0)
print(e._op, sorted(v.data for v in e._prev))   # + [-6.0, 10.0]
print(d._op, sorted(v.data for v in d._prev))   # * [-3.0, 2.0]

## 4. Le gradient entre dans la boîte : `_backward`

Deux nouveautés dans le constructeur : `grad`, la pente du résultat final par
rapport à ce nœud (démarre à 0.0), et `_backward`, le mode d'emploi du retour,
que chaque opération remplit à la naissance de sa sortie. Les dérivées
locales : l'**addition** transmet le gradient **tel quel** (pente 1) ; la
**multiplication** **échange les valeurs de ses opérandes** (dans
`coût = dose × prix`, la sensibilité à la dose est le prix, et réciproquement).
Attention : les `=` de cette version cachent un piège, révélé à la section 5.

In [ ]:
class Value:
    def __init__(self, data, _prev=(), _op=""):
        self.data = data
        self.grad = 0.0                     # la pente, inconnue pour l'instant
        self._backward = lambda: None       # le mode d'emploi du retour
        self._prev = set(_prev)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad = out.grad            # derivee locale 1 : transmis tel quel
            other.grad = out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad = other.data * out.grad    # l'echange : la valeur de l'autre
            other.grad = self.data * out.grad
        out._backward = _backward

        return out

    def __repr__(self):
        return f"Value(data={self.data})"

Le retour **à la main**, du sommet vers les feuilles : l'amorce `L.grad = 1.0`
(L bouge de 1 quand L bouge de 1), puis un `_backward` par nœud. Suis chaque
gradient à la trace, c'est le meilleur exercice du chapitre.

In [ ]:
a = Value(2.0); b = Value(-3.0); c = Value(10.0)
d = a * b          # -6.0
e = d + c          #  4.0
f = Value(-2.0)
L = e * f          # -8.0, notre " loss " miniature

L.grad = 1.0       # amorce : L bouge de 1 quand L bouge de 1
L._backward()      # remplit e.grad et f.grad
e._backward()      # remplit d.grad et c.grad
d._backward()      # remplit a.grad et b.grad

print(f"L {L.grad} | e {e.grad} | f {f.grad} | d {d.grad}")
print(f"c {c.grad} | a {a.grad} | b {b.grad}")

In [ ]:
# Le juge independant : le capteur du chapitre 3 confirme a.grad = 6.
# On refait tout le calcul en fonction de a, et on mesure la pente.
pente_capteur = pente(lambda av: ((av * -3.0) + 10.0) * -2.0, 2.0)
print(f"capteur sur a : {pente_capteur:.6f} | moteur : {a.grad}")

## 5. Le piège du nœud utilisé deux fois

`s = a + a = 2a` : la vraie pente vaut 2, sans discussion. Exécute et regarde
le moteur répondre 1.0, **sans aucune erreur**. Diagnostic : dans `_backward`,
`self` et `other` sont ici LE MÊME objet `a` ; le second `=` écrase le premier
message au lieu de s'y ajouter.

In [ ]:
# CAS QUI ECHOUE : le noeud utilise deux fois.
a = Value(3.0)
s = a + a          # a joue les DEUX roles de l'addition

s.grad = 1.0
s._backward()
print(f"moteur  : a.grad = {a.grad}")
print(f"capteur : {pente(lambda av: av + av, 3.0):.4f}")
print("Le moteur repond 1.0, la vraie pente vaut 2 : le second message a ECRASE le premier.")

La correction tient en deux caractères par ligne : `=` devient `+=`. Un nœud
utilisé sur plusieurs chemins reçoit plusieurs contributions, qui doivent
s'**additionner** ; voilà pourquoi `grad` démarre à 0.0 (un compteur), et
pourquoi il faudra le remettre à zéro avant chaque backward : c'est le
`zero_grad()` du chapitre 1. Le correctif entre dans la classe finale, juste
en dessous.

## 6. Compléter le moteur : tanh, puis le retour automatique

Deux derniers chantiers. `tanh`, la première opération « toute faite » : sa
dérivée locale est un bijou, si `t = tanh(x)` la pente vaut `1 - t**2`, lisible
sur la sortie elle-même. Et `backward()`, le chef d'orchestre : un **tri
topologique** (un nœud n'entre dans la liste qu'après toute sa généalogie),
l'amorce à 1.0, puis la liste parcourue à l'envers. Règle absolue : un nœud ne
distribue son gradient qu'une fois qu'il a reçu TOUTES ses contributions.

In [ ]:
import math


class Value:
    def __init__(self, data, _prev=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_prev)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad           # += : les contributions s'ADDITIONNENT
            other.grad += out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1 - t ** 2) * out.grad    # derivee locale : 1 - tanh**2
        out._backward = _backward

        return out

    def backward(self):
        ordre = []
        vus = set()

        def visiter(v):
            if v not in vus:
                vus.add(v)
                for parent in v._prev:
                    visiter(parent)
                ordre.append(v)
        visiter(self)

        self.grad = 1.0
        for v in reversed(ordre):
            v._backward()

    def __repr__(self):
        return f"Value(data={self.data})"

In [ ]:
# Rejoue le cas qui echouait : les contributions s'additionnent desormais.
a = Value(3.0)
s = a + a
s.grad = 1.0
s._backward()
print(f"a.grad = {a.grad}  (la vraie pente de s = 2a vaut bien 2)")

In [ ]:
# La derivee locale de tanh, 1 - tanh**2, confirmee par le capteur du chapitre 3 :
x = Value(0.5)
y = x.tanh()
y.grad = 1.0
y._backward()
print(f"formule 1 - t**2 : {1 - math.tanh(0.5) ** 2:.10f}")
print(f"capteur          : {pente(math.tanh, 0.5):.10f}")
print(f"ton moteur       : {x.grad:.10f}")

Baptême du feu : un **neurone** entier (deux entrées, deux poids, un biais, un
tanh), dérivé en une seule ligne. Lis les gradients : `w2.grad = 0` parce que
`x2 = 0`, un poids sans influence ici. Un gradient répond à la question
« qu'est-ce qui compte, là, maintenant ? ».

In [ ]:
x1, x2 = Value(1.0), Value(0.0)                  # les entrees
w1, w2 = Value(2.0), Value(1.0)                 # les poids
b = Value(-0.9013877113318902)                    # le biais

n = x1 * w1 + x2 * w2 + b                        # la somme ponderee : 1.0986
o = n.tanh()                                     # la sortie : 0.8000

o.backward()                                     # UNE ligne. C'est tout.
print(f"n = {n.data:.4f} | o = {o.data:.4f}")
print(f"w1.grad = {w1.grad:.4f} | w2.grad = {w2.grad:.4f} | b.grad = {b.grad:.4f}")
print(f"x1.grad = {x1.grad:.4f} | x2.grad = {x2.grad:.4f}")

## 7. Le gradient check : l'assurance qualité des labos

Le trajet de Sètondji (chapitre 3) : 8 km payés 1 250 FCFA, tarif candidat
`w = 100` FCFA/km, erreur au carré. Le moteur remonte le graphe ; le capteur
mesure la courbe. Deux machines indépendantes, un seul nombre : -7200. Note le
clin d'œil : `ecart * ecart` est le nœud partagé de la section 5 ; sans le
`+=`, le moteur répondrait -3600, moitié de la vérité, sans prévenir.

In [ ]:
w = Value(100.0)
pred = w * 8.0                    # le prix predit : 800 FCFA
ecart = pred + (-1250.0)          # l'ecart au prix reel : -450
err = ecart * ecart               # l'erreur au carre : 202500 (et le noeud partage !)

err.backward()
print(f"moteur  : w.grad = {w.grad}")


def erreur_trajet(w):
    return (w * 8.0 - 1250.0) ** 2


print(f"capteur : {pente(erreur_trajet, 100.0)}")

## 8. Un neurone entraîné avec TON moteur

Le refrain du livre (prédire, mesurer l'erreur, corriger, recommencer), sans
une ligne de PyTorch. Quatre élèves, deux notes chacun (ramenées entre 0 et 1
pour ne pas saturer tanh), cible admis (+1) ou recalé (-1). `perte()` fait
l'aller et construit un graphe **neuf** à chaque étape ; la remise à zéro des
grad joue le rôle du `optimiseur.zero_grad()` du chapitre 1.

In [ ]:
notes = [[0.9, 0.8], [0.3, 0.2], [0.8, 0.6], [0.2, 0.4]]   # (maths, francais), sur 1
cibles = [1.0, -1.0, 1.0, -1.0]                             # admis / recale

w1, w2, b = Value(0.1), Value(-0.2), Value(0.05)            # depart quelconque


def perte():
    total = Value(0.0)
    for (xm, xf), cible in zip(notes, cibles):
        o = (w1 * xm + w2 * xf + b).tanh()      # predire
        e = o + (-cible)
        total = total + e * e                   # mesurer l'erreur (au carre)
    return total


for etape in range(101):
    loss = perte()                          # l'aller construit le graphe
    if etape % 20 == 0:
        print(f"étape {etape:3d} | loss = {loss.data:.4f}")
    w1.grad = w2.grad = b.grad = 0.0        # remise a zero (le += s'accumule !)
    loss.backward()                         # le retour : TES gradients
    for p in (w1, w2, b):
        p.data -= 0.5 * p.grad              # corriger : la descente du chapitre 3

loss_finale = perte()
print(f"final     | loss = {loss_finale.data:.4f}")

In [ ]:
# Le neurone, interroge apres entrainement : quatre predictions justes.
for (xm, xf), cible in zip(notes, cibles):
    o = (w1 * xm + w2 * xf + b).tanh()
    print(f"notes ({xm}, {xf}) -> {o.data:+.3f}   (cible {cible:+.0f})")

## 9. La révélation : PyTorch fait exactement ça

Le même neurone en tenseurs `float64` : `requires_grad` marque les feuilles,
le graphe se construit pendant l'aller (`grad_fn` !), `.backward()` trie,
amorce et accumule. Les chiffres coïncident à 15 décimales (l'ultime bit
diffère : la tanh interne de PyTorch et celle du module `math` n'arrondissent
pas pareil leur dernier chiffre), et **bit pour bit** dès qu'il n'y a pas de
tanh.

In [ ]:
import torch

x1t = torch.tensor(1.0, dtype=torch.float64)
x2t = torch.tensor(0.0, dtype=torch.float64)
w1t = torch.tensor(2.0, dtype=torch.float64, requires_grad=True)
w2t = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)
bt = torch.tensor(-0.9013877113318902, dtype=torch.float64, requires_grad=True)

ot = torch.tanh(x1t * w1t + x2t * w2t + bt)
ot.backward()

# Le meme neurone, avec TON moteur (graphe neuf, gradients neufs) :
x1v, x2v = Value(1.0), Value(0.0)
w1v, w2v = Value(2.0), Value(1.0)
bv = Value(-0.9013877113318902)
ov = (x1v * w1v + x2v * w2v + bv).tanh()
ov.backward()

print(f"PyTorch    : {w1t.grad.item()} {w2t.grad.item()} {bt.grad.item()}")
print(f"ton moteur : {w1v.grad} {w2v.grad} {bv.grad}")
print(f"grad_fn de o (le ticket _backward de PyTorch) : {ot.grad_fn}")

# Accord a 15 decimales : seule la tanh interne differe d'un ultime bit d'arrondi.
assert abs(w1t.grad.item() - w1v.grad) < 1e-14
assert w2t.grad.item() == w2v.grad == 0.0
assert abs(bt.grad.item() - bv.grad) < 1e-14

In [ ]:
# Sur le graphe de Setondji (sans tanh), l'accord est BIT POUR BIT.
wt = torch.tensor(100.0, dtype=torch.float64, requires_grad=True)
errt = (wt * 8.0 - 1250.0) ** 2
errt.backward()

wv = Value(100.0)
predv = wv * 8.0
ecartv = predv + (-1250.0)
errv = ecartv * ecartv
errv.backward()

print(f"PyTorch    : {wt.grad.item()}")
print(f"ton moteur : {wv.grad}")
assert wt.grad.item() == wv.grad == -7200.0, "accord exact attendu"
print("Le meme algorithme, les memes nombres. La magie a cesse d'en etre une.")

### Alors pourquoi PyTorch ? Le grain, pas l'algorithme

Ton moteur crée un objet Python par **nombre** ; PyTorch crée un nœud par
**tenseur**, et exécute des millions d'opérations par nœud en C++ ou sur GPU.
Mesure honnête + estimation d'ordre de grandeur :

In [ ]:
import time

# Le grain de ton moteur : UN objet Python par nombre. Mesurons son debit.
N = 200_000
t0 = time.perf_counter()
acc = Value(0.0)
un = Value(1.0)
for _ in range(N // 2):
    acc = acc + un
    acc = acc * un
t1 = time.perf_counter()
par_op = (t1 - t0) / N
print(f"cout moyen mesure : {par_op * 1e6:.2f} microsecondes par operation Value")

# ESTIMATION d'ordre de grandeur (pas une mesure) : l'aller de MiniLM,
# c'est ~5.7 millions de multiplications (chapitre 2).
aller = par_op * 5.7e6
print(f"estimation : ~{aller:.0f} s par etape rien que pour l'aller,")
print(f"soit ~{aller * 8001 / 3600:.0f} h pour les 8001 etapes du chapitre 1 (sans compter le retour),")
print("la ou PyTorch, avec un noeud par TENSEUR et du code optimise, a tout fait en ~9 s.")

## Exercices

Le moment le plus important du livre : tu refermes la leçon, et tu écris TON
moteur. Six exercices qui suivent l'ordre de construction du chapitre (chaque
étape s'appuie sur la précédente) ; les niveaux ● à ●●● indiquent l'effort
attendu. Chaque cellule marquée `# TODO(toi)` contient un ou plusieurs trous ;
complète, puis exécute la cellule de validation (`assert`) qui suit : si elle
passe sans erreur, c'est gagné.

**Le pacte « IA débranchée » s'applique ici en entier** : ferme l'onglet de ton
assistant, littéralement. Si un trou résiste, relis la section correspondante
de la leçon, dessine le graphe sur papier, et insiste ; l'idéal est même
d'écrire chaque classe sans recopier la leçon. Les réponses sont dans le
notebook solution, à n'ouvrir qu'après avoir vraiment essayé.

### Exercice 1 · `Value`, version 1 : la boîte qui se souvient — niveau ●

`data` (la valeur), `_prev` (les parents, en ensemble), `_op` (l'opération qui
l'a produite). Les méthodes spéciales `__add__` et `__mul__` doivent calculer
ET déclarer la naissance, et accepter un opérande nu (`w * 8.0`) grâce au
`isinstance`.

In [ ]:
class Value:
    def __init__(self, data, _prev=(), _op=""):
        # TODO(toi) : range les trois attributs de la boite.
        # 1) self.data  <- data (le nombre lui-meme)
        # 2) self._prev <- set(_prev) (les parents, en ensemble)
        # 3) self._op   <- _op (le nom de l'operation qui l'a produite)
        ...

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        # TODO(toi) : calcule ET declare la naissance.
        # Renvoie une Value dont : data = self.data + other.data,
        # parents = (self, other), operation = "+".
        return ...

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        # TODO(toi) : pareil, pour la multiplication (_op = "*").
        return ...

    def __repr__(self):
        return f"Value(data={self.data})"

In [ ]:
# Validation : le calcul se fait ET s'enregistre.
a = Value(2.0); b = Value(-3.0); c = Value(10.0)
d = a * b        # -6.0, ne de a et b par "*"
e = d + c        #  4.0, ne de d et c par "+"
assert isinstance(e, Value), "e doit etre une Value, pas un float"
assert d.data == -6.0 and e.data == 4.0, "les data doivent valoir -6.0 et 4.0"
assert d._op == "*" and e._op == "+", "_op doit garder le nom de l'operation"
assert d._prev == {a, b}, "les parents de d sont a et b"
assert e._prev == {d, c}, "les parents de e sont d et c"
assert a._prev == set() and a._op == "", "une feuille n'a ni parents ni operation"
assert (a + 1.0).data == 3.0, "l'autre operande peut arriver nu (isinstance)"
print("Exercice 1 OK : chaque calcul laisse sa trace.")

### Exercice 2 · Les dérivées locales : `_backward` de `+` et `*` — niveau ●●

Le constructeur reçoit deux nouveautés (fournies) : `grad`, qui démarre à 0.0,
et `_backward`, le mode d'emploi du retour. À toi de l'écrire pour chaque
opération : l'**addition** transmet le gradient **tel quel**, la
**multiplication** **échange les valeurs de ses opérandes**. Écris-les avec
`=`, comme la version naïve de la leçon : le piège du nœud partagé sera déminé
à l'exercice 3, c'est voulu.

In [ ]:
class Value:
    def __init__(self, data, _prev=(), _op=""):
        self.data = data
        self.grad = 0.0                     # la pente, inconnue pour l'instant
        self._backward = lambda: None       # le mode d'emploi du retour
        self._prev = set(_prev)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            # TODO(toi) : l'addition transmet le gradient tel quel
            # (derivee locale 1). Ecris, pour self PUIS pour other :
            #   grad = out.grad
            ...
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            # TODO(toi) : la multiplication ECHANGE les valeurs des operandes.
            # self recoit  out.grad x la valeur de other (other.data),
            # other recoit out.grad x la valeur de self  (self.data).
            ...
        out._backward = _backward

        return out

    def __repr__(self):
        return f"Value(data={self.data})"

In [ ]:
# Validation : le retour a la main, sept gradients exacts.
a = Value(2.0); b = Value(-3.0); c = Value(10.0)
d = a * b          # -6.0
e = d + c          #  4.0
f = Value(-2.0)
L = e * f          # -8.0, notre " loss " miniature

L.grad = 1.0       # amorce : L bouge de 1 quand L bouge de 1
L._backward()      # remplit e.grad et f.grad
e._backward()      # remplit d.grad et c.grad
d._backward()      # remplit a.grad et b.grad

assert L.grad == 1.0, "l'amorce : L.grad = 1.0"
assert e.grad == -2.0 and f.grad == 4.0, "L = e*f : l'echange des valeurs"
assert d.grad == -2.0 and c.grad == -2.0, "e = d+c : transmis tel quel"
assert a.grad == 6.0 and b.grad == -4.0, "d = a*b : l'echange, encore"

# Le juge independant : le capteur du chapitre 3, sur a.
pente_capteur = pente(lambda av: ((av * -3.0) + 10.0) * -2.0, 2.0)
assert abs(pente_capteur - a.grad) < 1e-3, "le capteur doit confirmer a.grad = 6"
print(f"Exercice 2 OK. Capteur sur a : {pente_capteur:.6f} (moteur : {a.grad})")

### Exercice 3 · La classe finale : accumulation, `tanh`, `backward()` — niveau ●●●

Trois chantiers pour clore le moteur :

- **l'accumulation** : un nœud utilisé sur plusieurs chemins reçoit plusieurs
  contributions, qui doivent s'**additionner** : `+=` partout dans les
  `_backward` (voilà pourquoi `grad` démarre à 0.0) ;
- **`tanh`** : ta première opération « toute faite ». Si `t = tanh(x)`, la
  dérivée locale vaut `1 - t**2`, lisible sur la sortie elle-même ;
- **`backward()`** : le tri topologique (un nœud n'entre dans la liste
  qu'APRÈS toute sa généalogie), puis l'amorce à 1.0 et la liste parcourue à
  l'envers. Règle absolue : un nœud ne distribue son gradient qu'une fois
  qu'il a reçu TOUTES ses contributions.

In [ ]:
import math


class Value:
    def __init__(self, data, _prev=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_prev)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            # TODO(toi) : reprends ta version de la section 2, mais avec +=
            # (les contributions s'ADDITIONNENT au lieu de s'ecraser).
            ...
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            # TODO(toi) : l'echange des valeurs, avec += aussi.
            ...
        out._backward = _backward

        return out

    def tanh(self):
        # TODO(toi) : la premiere operation " toute faite " du moteur.
        # 1) t <- math.tanh(self.data)
        # 2) out <- Value(t, (self,), "tanh")   (un seul parent !)
        # 3) _backward : self.grad += (1 - t**2) * out.grad
        # 4) accroche _backward a out, puis renvoie out
        ...

    def backward(self):
        ordre = []
        vus = set()

        def visiter(v):
            # TODO(toi) : le tri topologique, en recursif.
            # Si v n'a pas encore ete vu : marque-le dans vus, visite chacun
            # de ses parents (v._prev), PUIS ajoute v a ordre (apres eux).
            ...
        visiter(self)

        # TODO(toi) : le retour automatique.
        # 1) amorce : self.grad = 1.0
        # 2) pour chaque noeud v de ordre parcouru A L'ENVERS (reversed),
        #    appelle v._backward()
        ...

    def __repr__(self):
        return f"Value(data={self.data})"

In [ ]:
# Validation en trois temps : accumulation, tanh, backward automatique.
# 1) Le piege du noeud partage est demine.
a = Value(3.0)
s = a + a
s.grad = 1.0
s._backward()
assert a.grad == 2.0, f"a.grad doit valoir 2.0 (1.0 + 1.0), obtenu {a.grad}"

# 2) tanh : la valeur, et la derivee locale 1 - tanh**2.
x = Value(0.5)
y = x.tanh()
assert abs(y.data - math.tanh(0.5)) < 1e-12, "tanh doit utiliser math.tanh"
assert y._prev == {x} and y._op == "tanh", "un seul parent, _op = 'tanh'"
y.grad = 1.0
y._backward()
exacte = 1 - math.tanh(0.5) ** 2
assert abs(x.grad - exacte) < 1e-9, f"derivee locale attendue {exacte}, obtenue {x.grad}"

# 3) Le neurone entier, derive en UNE ligne : cinq gradients exacts.
x1, x2 = Value(1.0), Value(0.0)
w1, w2 = Value(2.0), Value(1.0)
b = Value(-0.9013877113318902)
o = (x1 * w1 + x2 * w2 + b).tanh()
o.backward()
assert abs(o.data - 0.8) < 1e-4, "la sortie doit valoir 0.8"
assert abs(w1.grad - 0.36) < 1e-6, f"w1.grad attendu 0.36, obtenu {w1.grad}"
assert w2.grad == 0.0, "w2 multiplie x2 = 0 : aucune influence, gradient nul"
assert abs(b.grad - 0.36) < 1e-6, f"b.grad attendu 0.36, obtenu {b.grad}"
assert abs(x1.grad - 0.72) < 1e-6, f"x1.grad attendu 0.72, obtenu {x1.grad}"
assert abs(x2.grad - 0.36) < 1e-6, f"x2.grad attendu 0.36, obtenu {x2.grad}"
print("Exercice 3 OK : ton moteur est complet.")

### Exercice 4 · Le gradient check de Sètondji — niveau ●●

Le trajet du chapitre 3 : 8 km payés 1 250 FCFA, tarif candidat `w = 100`
FCFA/km, erreur au carré. Construis le graphe avec TES `Value` et lance le
backward : ton moteur et le capteur doivent tomber d'accord sur -7200. Note le
clin d'œil : `ecart * ecart` est le nœud partagé de l'exercice 3 ; sans ton
`+=`, le moteur répondrait -3600, moitié de la vérité.

In [ ]:
w = Value(100.0)
# TODO(toi) : construis le graphe de l'erreur de Setondji avec TES Value.
# pred  <- w * 8.0            (le prix predit, 800 FCFA)
# ecart <- pred + (-1250.0)   (l'ecart au prix reel, -450)
# err   <- ecart * ecart      (l'erreur au carre : le noeud partage !)
# puis lance err.backward()
pred = ...
ecart = ...
err = ...

print(f"moteur  : w.grad = {w.grad}")


def erreur_trajet(w):
    return (w * 8.0 - 1250.0) ** 2


print(f"capteur : {pente(erreur_trajet, 100.0)}")

In [ ]:
# Validation par gradient check : deux machines independantes, un seul nombre.
assert err.data == 202500.0, "l'erreur au carre doit valoir 202500"
assert w.grad == -7200.0, f"le moteur doit repondre -7200.0 exactement, obtenu {w.grad}"
assert abs(w.grad - pente(erreur_trajet, 100.0)) < 1e-3, "moteur et capteur doivent coincider"
print("Exercice 4 OK : ton moteur est valide, comme dans les labos.")

### Exercice 5 · Un neurone entraîné avec TON moteur — niveau ●●

Le refrain du livre (prédire, mesurer l'erreur, corriger, recommencer), sans
une ligne de PyTorch. `perte()` est fournie : elle fait l'aller et construit un
graphe **neuf** à chaque étape. À toi le cœur de la boucle : remise à zéro des
gradients, backward, descente.

In [ ]:
notes = [[0.9, 0.8], [0.3, 0.2], [0.8, 0.6], [0.2, 0.4]]   # (maths, francais), sur 1
cibles = [1.0, -1.0, 1.0, -1.0]                             # admis / recale

w1, w2, b = Value(0.1), Value(-0.2), Value(0.05)            # depart quelconque


def perte():
    total = Value(0.0)
    for (xm, xf), cible in zip(notes, cibles):
        o = (w1 * xm + w2 * xf + b).tanh()      # predire
        e = o + (-cible)
        total = total + e * e                   # mesurer l'erreur (au carre)
    return total


for etape in range(101):
    loss = perte()                          # l'aller construit le graphe
    if etape % 20 == 0:
        print(f"étape {etape:3d} | loss = {loss.data:.4f}")
    # TODO(toi) : le refrain, avec TON moteur.
    # 1) remets a zero les gradients : w1.grad = w2.grad = b.grad = 0.0
    # 2) le retour : loss.backward()
    # 3) corrige chaque poids p parmi (w1, w2, b) : p.data -= 0.5 * p.grad
    ...

loss_finale = perte()
print(f"final     | loss = {loss_finale.data:.4f}")

In [ ]:
# Validation de l'entrainement : la loss a fondu, les predictions sont justes.
assert loss_finale.data < 0.05, f"loss finale attendue < 0.05, obtenue {loss_finale.data:.4f}"
for (xm, xf), cible in zip(notes, cibles):
    o = (w1 * xm + w2 * xf + b).tanh()
    assert o.data * cible > 0, f"prediction du mauvais signe pour ({xm}, {xf})"
    print(f"notes ({xm}, {xf}) -> {o.data:+.3f}   (cible {cible:+.0f})")
print("Exercice 5 OK : ton moteur vient de faire apprendre une machine.")

### Exercice 6 · `exp` rejoint le moteur — niveau ●●●

La leçon l'a promis : toute fonction dont tu connais la dérivée locale peut
rejoindre le moteur en huit lignes, sur le moule de `tanh`. Preuve par
l'**exponentielle**, dont le chapitre 5 aura besoin. Sa dérivée locale est
encore plus simple que celle de tanh : la dérivée de `exp(x)`, c'est `exp(x)`
elle-même, déjà calculée dans la sortie. Écris `exp(v)` en fonction (elle
prend une `Value`, elle en renvoie une), sans toucher à la classe.

In [ ]:
def exp(v):
    # TODO(toi) : l'exponentielle, sur le moule de tanh.
    # 1) e <- math.exp(v.data)
    # 2) out <- Value(e, (v,), "exp")
    # 3) _backward : v.grad += e * out.grad   (la derivee locale d'exp, c'est exp)
    # 4) accroche _backward a out, puis renvoie out
    ...


In [ ]:
# Validation : valeur, graphe, et gradient check contre le capteur.
x = Value(1.3)
y = exp(x)
assert isinstance(y, Value), "exp doit renvoyer une Value"
assert abs(y.data - math.exp(1.3)) < 1e-12, "exp doit utiliser math.exp"
assert y._prev == {x} and y._op == "exp", "un seul parent, _op = 'exp'"
y.backward()
assert abs(x.grad - math.exp(1.3)) < 1e-9, "la derivee locale d'exp est exp elle-meme"
assert abs(x.grad - pente(math.exp, 1.3)) < 1e-3, "le capteur doit confirmer"
print(f"Exercice 6 OK : exp a rejoint ton moteur (x.grad = {x.grad:.6f}).")

## Et maintenant ?

Si les six validations passent, tu viens d'écrire, seul, le moteur qui se
cache derrière le `loss.backward()` du chapitre 1. Relis la ligne dans le
notebook du chapitre 1 : elle t'appartient. Au chapitre 5, les trois dernières
lignes opaques tombent (softmax, cross-entropy, échantillonnage) ; au
chapitre 6, tu branches des neurones comme le tien en réseau. Et rendez-vous au
chapitre 9 pour le second passage IA débranchée : l'attention.